In [1]:
using CMPSExcitations

In [142]:
# canonical basis
function projection_matrix1(D, R)
    Dr, M = eigen(R)

    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E
        W2 = E + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

function projection_matrix2(D, R)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    Dr, M = eigen(R)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        dD1 = view(e, 1:D)
        dD2 = view(e, D+1:2*D)
        X = zeros(D, D)

        k = 2D + 1
        for i in 1:D, j in 1:D
            if i != j
                X[i, j] = e[k] # sets the one hot vector
                k += 1
            end
        end

        Dr = Diagonal(Dr)
        W1 = M * ((X * Dr - Dr * X) + Diagonal(dD1)) / M
        W2 = M * ((X * Dr - Dr * X) + Diagonal(dD2)) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

function projection_matrix3(D, R)
    Dr, M = eigen(R)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E - M * Diagonal(F) / M
        W2 = E + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

# canonical basis
function excitation_matrix(Heff, D)
    dim = 2 * D^2
    M = zeros(ComplexF64, dim, dim)

    for j in 1:dim
        e = zeros(ComplexF64, dim)
        e[j] = 1.0
        W1 = reshape(view(e, 1:D^2), D, D)
        W2 = reshape(view(e, D^2+1:dim), D, D)
        W1p, W2p = Heff((Constant(W1), Constant(W2)))
        M[:, j] = vcat(vec(W1p[]), vec(W2p[]))
    end

    return M
end

excitation_matrix (generic function with 1 method)

In [172]:
Hsingle_ll(c, μ) = ∫(∂ψ̂' * ∂ψ̂ - μ * ψ̂' * ψ̂ + c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hsingle(c, μ) = ∫(2 * ∂ψ̂' * ∂ψ̂ - 2 * μ * ψ̂' * ψ̂ + 4 * c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hcoupled(c, μ) = ∫(
    (∂ψ̂₁' * ∂ψ̂₁ - μ * ψ̂₁' * ψ̂₁ + c * (ψ̂₁')^2 * ψ̂₁^2 +
     ∂ψ̂₂' * ∂ψ̂₂ - μ * ψ̂₂' * ψ̂₂ + c * (ψ̂₂')^2 * ψ̂₂^2 +
     2 * c * (ψ̂₁') * (ψ̂₂') * ψ̂₂ * ψ̂₁), (-Inf, +Inf));

In [173]:
c, μ = 10., 5.
tol = 1e-10

Ds = [4, 8, 12]
D = maximum(Ds)

HLL = Hsingle(c, μ)
@time stateLL = find_groundstate(Ds, HLL, YangGaudinCMPS, optalg=LBFGS(80; verbosity=1, maxiter=7000, gradtol=tol), gradtol=tol)
println("Energy density: ", expval(HLL.h, stateLL)[], "\n Particle density: ", expval(ψ̂' * ψ̂, stateLL)[], "\n Order parameter: ", expval(ψ̂, stateLL)[])

# -----
stateCLL = InfiniteCMPS(stateLL.Q, (stateLL.Rs[1], stateLL.Rs[1]));
HCLL = Hcoupled(c, μ)
println("Energy density: ", expval(HCLL.h, stateCLL)[], "\nParticle density: ", expval(ψ̂₁' * ψ̂₁ + ψ̂₂' * ψ̂₂, stateCLL)[], "\nDensity imbalance: ", expval(ψ̂₁' * ψ̂₁ - ψ̂₂' * ψ̂₂, stateCLL)[])

Optimizing D=4


┌ Info: YangGaudinCMPS ground state: initialization with e = 51.574758488621
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:106
┌ Info: LBFGS: converged after 107 iterations: f = -2.734747817523, ‖∇f‖ = 4.9454e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 4 | YangGaudinCMPS{Constant{Matrix{Float64}}, 1}
  0.064729 seconds (413.93 k allocations: 19.545 MiB)
---------------
Optimizing D=8


┌ Info: YangGaudinCMPS ground state: converged after 108 iterations: e = -2.734747817523, ‖∇e‖ = 4.9454e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:118
┌ Info: YangGaudinCMPS ground state: initialization with e = -2.734747817543
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:106
┌ Info: LBFGS: converged after 360 iterations: f = -2.761265509087, ‖∇f‖ = 9.6662e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 8 | YangGaudinCMPS{Constant{Matrix{Float64}}, 1}
  3.122784 seconds (3.52 M allocations: 323.298 MiB, 1.35% gc time)
---------------
Optimizing D=12


┌ Info: YangGaudinCMPS ground state: converged after 361 iterations: e = -2.761265509087, ‖∇e‖ = 9.6662e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:118
┌ Info: YangGaudinCMPS ground state: initialization with e = -2.761265509015
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:106
┌ Info: LBFGS: converged after 472 iterations: f = -2.764113438366, ‖∇f‖ = 7.5919e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 12 | YangGaudinCMPS{Constant{Matrix{Float64}}, 1}
 20.004298 seconds (9.92 M allocations: 1.537 GiB, 1.01% gc time)
---------------
 23.191986 seconds (13.85 M allocations: 1.872 GiB, 1.05% gc time)
Energy density: -2.764113438365665

┌ Info: YangGaudinCMPS ground state: converged after 473 iterations: e = -2.764113438366, ‖∇e‖ = 7.5919e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:118



 Particle density: 0.4368862360532374
 Order parameter: 0.32821394010773786
Energy density: -2.76411343836569
Particle density: 0.87377247210647
Density imbalance: 0.0


In [174]:
leftgauge!(stateCLL)

(InfiniteCMPS{Constant{Matrix{Float64}}, 2}(Constant{Matrix{Float64}}(Base.RefValue{Matrix{Float64}}([-0.6987384726953465 0.6007201763845302 … -0.9985365393768765 1.0174663466110996; -0.2867365689241142 -0.9962376317459747 … 0.0942585337932423 -0.5805423852776124; … ; -0.00694059050045076 0.030516775906827318 … -2.1861086704426085 -0.0897321793447293; -0.00023892071248943217 -0.026344866002920908 … -0.14215777620466644 -2.213635286555528])), (Constant{Matrix{Float64}}(Base.RefValue{Matrix{Float64}}([0.701614574848957 -0.328869346740481 … 0.6036749893516336 -1.0414245405530262; -0.19950718243189514 -0.6588045664090577 … -0.8421198403118489 -0.361118623920258; … ; -0.0024380999338953625 -0.007332198060133021 … -0.10747929443263013 0.32472736622827586; 0.006794121315924817 0.007630132563603347 … -0.012338604198076687 -0.14024774677663687])), Constant{Matrix{Float64}}(Base.RefValue{Matrix{Float64}}([0.701614574848957 -0.328869346740481 … 0.6036749893516336 -1.0414245405530262; -0.199507182

In [175]:
# common setup
p = 0 # momentum
R = stateCLL.Rs[1][]
D = size(R, 1) # R = MDᵣ/M
space = InfiniteCMPSExcitationSpace(p, stateCLL, stateCLL)
H = excitation_matrix(excitation_operator(HCLL, space), D);

In [176]:
P = projection_matrix1(D, R)
vals, vecs = eigen(P' * H * P, P' * P)
println(real.(vals[1:5]))

[-0.6806584198764069, 0.5037861743729247, 1.3335933331358512, 2.1458969325253032, 2.708621974888034]


In [177]:
P = Matrix(qr(projection_matrix1(D, R)).Q)
vals, vecs = eigen(P' * H * P)
println(real.(vals[1:5]))

[-0.6806584198763104, 0.5037861743728337, 1.3335933331360657, 2.145896932525117, 2.7086219748878384]


In [178]:
P = projection_matrix2(D, R)
vals, vecs = eigen(P' * H * P, P' * P)
println(real.(vals[1:5]))

[-0.6806584248666031, 0.5037861320826795, 1.3335933141766334, 2.1458969177846794, 2.7086219368085964]


In [179]:
P = Matrix(qr(projection_matrix2(D, R)).Q)
@assert P' * P ≈ I
vals, vecs = eigen(P' * H * P)
println(real.(vals[1:5]))

[-0.6806584198763815, 0.5037861743724936, 1.3335933331363379, 2.1458969325255746, 2.70862197488748]


In [180]:
P = projection_matrix3(D, R)
vals, vecs = eigen(P' * H * P, P' * P)
println(real.(vals[1:5]))

[-0.6806584198763046, 0.5037861743728835, 1.333593333136084, 2.1458969325252433, 2.708621974887973]


In [181]:
P = Matrix(qr(projection_matrix3(D, R)).Q)
@assert P' * P ≈ I
vals, vecs = eigen(P' * H * P)
println(real.(vals[1:5]))

[-0.6806584198763428, 0.5037861743728854, 1.3335933331360712, 2.145896932525274, 2.7086219748879516]


QR instead of geneigsolve seems to be consistently equivalent as expected. Different parametrization seem to agree as well. However, the same parametrization gives different results based on the ground state which presumably only varies by gauge.. ??? Specifically the projector seems to be the issue.